In [ ]:
# ============================================================
# 0. PREPARAÇÃO NO GOOGLE COLAB (rode só uma vez por sessão)
# ============================================================
# Antes: suba para a pasta "EcgArritmia" do seu Google Drive a pasta
#   sinais_ecg_4c_compacto  (gerada por 09_compactar_sinais.py)
# Ambiente de execução: GPU (T4).

from google.colab import drive
import os
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

PASTA_DRIVE = '/content/drive/MyDrive/EcgArritmia'
NOME_DATASET = 'sinais_ecg_4c'   # sinais 12 x 5000 (500 Hz), gerados por 08_gerar_sinais.py

# Copia a versão compactada para o disco local do Colab
# (ler direto do Drive a cada batch deixaria o treino muito lento)
!cp -r "{PASTA_DRIVE}/{NOME_DATASET}_compacto" /content/
%cd /content

# Remonta os X_*.npy (float16, em mV), desfazendo o 09_compactar_sinais.py:
# descomprime cada bloco, soma as diferenças (cumsum) e volta de µV para mV
import io
import lzma
import shutil
import numpy as np
from pathlib import Path

compacto = Path(f"{NOME_DATASET}_compacto")
destino = Path(NOME_DATASET)
destino.mkdir(exist_ok=True)

shutil.copy(compacto / "classes_9c.json", destino)

for conjunto in ["treino", "validacao", "teste"]:
    # y_*_9c.npy (rótulos originais) e classes_9c.json são usados nos modelos 2 e 3
    for arquivo in [f"y_{conjunto}.npy", f"registros_{conjunto}.csv",
                    f"y_{conjunto}_9c.npy", f"registros_{conjunto}_9c.csv"]:
        shutil.copy(compacto / arquivo, destino)
    if (destino / f"X_{conjunto}.npy").exists():
        print(f"X_{conjunto}.npy já remontado")
        continue
    n = len(np.load(destino / f"y_{conjunto}.npy"))
    X = np.lib.format.open_memmap(destino / f"X_{conjunto}.npy.tmp", mode="w+",
                                  dtype=np.float16, shape=(n, 12, 5000))
    inicio = 0
    for bloco in sorted(compacto.glob(f"X_{conjunto}_*.npy.xz")):
        d = np.load(io.BytesIO(lzma.decompress(bloco.read_bytes())))
        X[inicio:inicio + len(d)] = (np.cumsum(d, axis=2, dtype=np.int32) * 0.001).astype(np.float16)
        inicio += len(d)
    assert inicio == n, f"{conjunto}: {inicio} exames nos blocos, {n} rótulos"
    X.flush()
    del X
    (destino / f"X_{conjunto}.npy.tmp").rename(destino / f"X_{conjunto}.npy")
    print(f"X_{conjunto}.npy: {n} exames")

!ls -lh {NOME_DATASET}

In [ ]:
# ============================================================
# CNN 1D
# Classificação de arritmias direto no sinal de ECG (12 derivações)
# Base: PhysioNet "A large scale 12-lead ECG database for arrhythmia study"
# ============================================================

import torch
import torch.nn as nn
import torch.optim as optim
from pathlib import Path
import copy
import json
import time

import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader

import matplotlib.pyplot as plt

In [ ]:
# ============================================================
# 1. CONFIGURAÇÕES  (iguais às dos treinos em imagem)
# ============================================================

CAMINHO_DATASET = Path(NOME_DATASET)   # definido na célula 0

ARQUITETURA = "resnet1d"
AUGMENTATION = "A_sem_aug"       # sem augmentation, para comparar com a config A das imagens

# ------------------------------------------------------------
# TAREFA: escolha qual dos 3 modelos será treinado nesta execução
#   "modelo1": AFIB, GSVT, SB, SR   (classes agrupadas, como antes)
#   "modelo2": AFIB x AF            (especialista do grupo AFIB)
#   "modelo3": SVT, AT, AVRT, ST    (especialista do grupo GSVT)
#   "modelo4": SR x SA              (opcional: especialista do grupo SR; ver grupo_4c)
# ------------------------------------------------------------
TAREFA = "modelo1"

# Rótulos originais (9 classes), na ordem codificada nos y_*_9c.npy (lida do classes_9c.json)
def ler_classes_finas():
    dados = json.load(open(CAMINHO_DATASET / "classes_9c.json", encoding="utf-8"))
    if isinstance(dados, dict) and "classes" in dados:
        dados = dados["classes"]
    if isinstance(dados, dict):
        if all(isinstance(v, int) for v in dados.values()):      # nome -> índice
            return [n for n, _ in sorted(dados.items(), key=lambda kv: kv[1])]
        return [dados[k] for k in sorted(dados, key=int)]        # índice -> nome
    return list(dados)


CLASSES_FINAS = ler_classes_finas()
print("Classes originais (9c):", CLASSES_FINAS)

# Para cada tarefa: classe -> rótulos finos que a compõem (a ordem define o índice da classe)
TAREFAS = {
    "modelo1": {"AFIB": ["AFIB", "AF"],
                "GSVT": ["SVT", "AT", "AVRT", "ST"],
                "SB": ["SB"],
                "SR": ["SR", "SA"]},          # no grupo_4c, SA pertence ao SR
    "modelo2": {"AFIB": ["AFIB"], "AF": ["AF"]},
    "modelo3": {"SVT": ["SVT"], "AT": ["AT"], "AVRT": ["AVRT"], "ST": ["ST"]},
    "modelo4": {"SR": ["SR"], "SA": ["SA"]},
}

CLASSES = list(TAREFAS[TAREFA].keys())

# Confere se todos os rótulos usados nas tarefas existem no classes_9c.json
_usados = {f for t in TAREFAS.values() for finas in t.values() for f in finas}
assert _usados <= set(CLASSES_FINAS), f"Rótulos fora do classes_9c.json: {_usados - set(CLASSES_FINAS)}"

BATCH_SIZE = 32
NUM_EPOCHS = 100
LEARNING_RATE = 0.001
PATIENCE = 5

torch.manual_seed(42)

# Tudo deste experimento é salvo no Drive (o disco do Colab é apagado ao fim da sessão)
PASTA_RESULTADOS = Path(PASTA_DRIVE) / "resultados"
PASTA_RESULTADOS.mkdir(exist_ok=True)
PREFIXO = f"{NOME_DATASET}_{TAREFA}_{ARQUITETURA}_{AUGMENTATION}"

# Verifica se existe GPU disponível
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Dispositivo utilizado:", device)
print("Experimento:", TAREFA, "-", ARQUITETURA, "-", AUGMENTATION)
print("Classes:", CLASSES)

In [ ]:
# ============================================================
# 2. CARREGAMENTO DO DATASET
# ============================================================

# A divisão treino/validação/teste é a mesma das imagens
# (08_gerar_sinais.py lê o divisao.csv do dataset_ecg_4c).
#
# Cada exame é uma matriz 12 x 5000 em mV. Aqui cada derivação é
# normalizada (média 0, desvio 1), o papel que o Normalize tinha nas imagens.

def carregar_rotulos_finos(conjunto, n):
    """Rótulos originais (índices em CLASSES_FINAS), um por exame, na mesma ordem do X_*.npy
    (arquivo y_<conjunto>_9c.npy)."""
    fino = np.load(CAMINHO_DATASET / f"y_{conjunto}_9c.npy", allow_pickle=True)
    if fino.dtype.kind in "OUS":                              # texto -> índice em CLASSES_FINAS
        fino = np.array([CLASSES_FINAS.index(str(v)) for v in fino])
    assert len(fino) == n, f"{conjunto}: {len(fino)} rótulos finos para {n} exames"
    return fino.astype(int)


def preparar_rotulos(conjunto):
    """Devolve (índices dos exames usados, rótulos da tarefa) para o conjunto."""
    y4 = np.load(CAMINHO_DATASET / f"y_{conjunto}.npy")       # 4 classes: AFIB, GSVT, SB, SR
    if TAREFA == "modelo1":
        return np.arange(len(y4)), y4                         # igual ao treino anterior

    fino = carregar_rotulos_finos(conjunto, len(y4))

    # Confere se os rótulos finos batem com o y de 4 classes (ordem de codificação correta)
    para_4c = {}
    for i, (_, finas) in enumerate(TAREFAS["modelo1"].items()):
        for f in finas:
            para_4c[CLASSES_FINAS.index(f)] = i
    previsto_4c = np.array([para_4c[int(v)] for v in fino])
    if not (previsto_4c == y4).all():
        print(f"[{conjunto}] divergentes: {(previsto_4c != y4).sum()} de {len(y4)} exames")
        print("Linhas = rótulo original (9c) | colunas = índice no y de 4 classes")
        print(pd.crosstab(pd.Series([CLASSES_FINAS[int(v)] for v in fino], name="9c"),
                          pd.Series(y4, name="y_4c")))
        raise AssertionError(f"{conjunto}: rótulos finos não correspondem ao y de 4 classes")

    # Mantém só os exames das classes da tarefa e remapeia para 0..N-1
    mapa = {}
    for novo, finas in enumerate(TAREFAS[TAREFA].values()):
        for f in finas:
            mapa[CLASSES_FINAS.index(f)] = novo
    indices = np.array([i for i, v in enumerate(fino) if int(v) in mapa])
    return indices, np.array([mapa[int(fino[i])] for i in indices])


class SinaisECG(Dataset):
    def __init__(self, conjunto):
        # mmap: lê do disco só os exames de cada batch, sem carregar ~3,6 GB na RAM
        self.X = np.load(CAMINHO_DATASET / f"X_{conjunto}.npy", mmap_mode="r")
        self.indices, self.y = preparar_rotulos(conjunto)     # filtra os exames da tarefa

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        x = self.X[self.indices[i]].astype(np.float32)        # (12, 5000)
        x = (x - x.mean(axis=1, keepdims=True)) / (x.std(axis=1, keepdims=True) + 1e-6)
        return torch.from_numpy(x), int(self.y[i])


dataset_treino = SinaisECG("treino")
dataset_validacao = SinaisECG("validacao")
dataset_teste = SinaisECG("teste")

print("Formato de um exame:", tuple(dataset_treino[0][0].shape))
print("Exames de treino:", len(dataset_treino))
print("Exames de validação:", len(dataset_validacao))
print("Exames de teste:", len(dataset_teste))

# Quantidade de exames por classe (sem cópias: não há augmentation)
contagem_classes = {}
for indice, classe in enumerate(CLASSES):
    n_treino = int((dataset_treino.y == indice).sum())
    contagem_classes[classe] = {
        "treino_originais": n_treino,
        "treino_copias": 0,
        "treino_total": n_treino,
        "validacao": int((dataset_validacao.y == indice).sum()),
        "teste": int((dataset_teste.y == indice).sum()),
    }

print()
print(pd.DataFrame(contagem_classes).T)

In [ ]:
# ============================================================
# 3. DATALOADERS
# ============================================================

treino_loader = DataLoader(dataset_treino, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
validacao_loader = DataLoader(dataset_validacao, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
teste_loader = DataLoader(dataset_teste, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

**ResNet1D**: a mesma ideia da ResNet, mas com convoluções 1D, que deslizam ao longo do tempo. As 12 derivações entram como 12 canais (como o R, G e B de uma imagem).

- Não há pesos pré-treinados (o ImageNet é de imagens), então **a rede inteira é treinada do zero** e nada é congelado.
- A primeira convolução tem kernel 31 (62 ms a 500 Hz). Depois dela e do pooling, o sinal fica com 1250 pontos, e cada estágio seguinte o reduz pela metade: 1250 → 625 → 313 → 157.
- No fim, a média no tempo (`AdaptiveAvgPool1d`) gera um vetor de 512 valores, que a camada linear transforma nas classes da tarefa escolhida (4, 2, 4 ou 2).

In [ ]:
# ============================================================
# 4. MODELO: ResNet1D
# ============================================================

class BlocoResidual(nn.Module):
    def __init__(self, canais_entrada, canais_saida, passo):
        super().__init__()
        self.convolucoes = nn.Sequential(
            nn.Conv1d(canais_entrada, canais_saida, kernel_size=7, stride=passo, padding=3, bias=False),
            nn.BatchNorm1d(canais_saida),
            nn.ReLU(),
            nn.Conv1d(canais_saida, canais_saida, kernel_size=7, stride=1, padding=3, bias=False),
            nn.BatchNorm1d(canais_saida),
        )
        # Atalho: soma a entrada na saída (ajustando o tamanho quando muda)
        if passo == 1 and canais_entrada == canais_saida:
            self.atalho = nn.Identity()
        else:
            self.atalho = nn.Sequential(
                nn.Conv1d(canais_entrada, canais_saida, kernel_size=1, stride=passo, bias=False),
                nn.BatchNorm1d(canais_saida),
            )

    def forward(self, x):
        return torch.relu(self.convolucoes(x) + self.atalho(x))


class ResNet1D(nn.Module):
    def __init__(self, numero_classes):
        super().__init__()
        self.entrada = nn.Sequential(
            nn.Conv1d(12, 64, kernel_size=31, stride=2, padding=15, bias=False),   # 5000 -> 2500
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=3, stride=2, padding=1),                      # 2500 -> 1250
        )
        self.blocos = nn.Sequential(
            BlocoResidual(64, 64, 1),   BlocoResidual(64, 64, 1),     # 1250
            BlocoResidual(64, 128, 2),  BlocoResidual(128, 128, 1),   # 625
            BlocoResidual(128, 256, 2), BlocoResidual(256, 256, 1),   # 313
            BlocoResidual(256, 512, 2), BlocoResidual(512, 512, 1),   # 157
        )
        self.saida = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(512, numero_classes),
        )

    def forward(self, x):            # x: (batch, 12, 5000)
        return self.saida(self.blocos(self.entrada(x)))


numero_classes = len(CLASSES)
modelo = ResNet1D(numero_classes).to(device)

print(f"Parâmetros treináveis: {sum(p.numel() for p in modelo.parameters()):,}")
with torch.no_grad():
    print("Saída para 1 exame:", tuple(modelo(torch.zeros(1, 12, 5000, device=device)).shape))

In [ ]:
# ============================================================
# 5. FUNÇÃO DE PERDA
# ============================================================

# CrossEntropyLoss é adequada para classificação multiclasse
# (sem pesos por classe, como nos treinos em imagem)

criterio = nn.CrossEntropyLoss()

# ============================================================
# 6. OTIMIZADOR
# ============================================================

# Diferente do transfer learning: todos os parâmetros são treinados.

otimizador = optim.Adam(
    modelo.parameters(),
    lr=LEARNING_RATE
)

# ============================================================
# 7. LISTAS PARA ARMAZENAR OS RESULTADOS
# ============================================================

historico_loss_treino = []
historico_loss_validacao = []

historico_acc_treino = []
historico_acc_validacao = []

inicio_treino = time.time()

In [ ]:
# ============================================================
# 8. TREINAMENTO
# ============================================================
melhor_val_loss = float('inf')

melhor_modelo = None

epocas_sem_melhora = 0

melhor_epoca = 0

for epoca in range(NUM_EPOCHS):

    # --------------------------------------------------------
    # TREINO
    # --------------------------------------------------------

    modelo.train()

    loss_treino = 0
    acertos_treino = 0
    total_treino = 0

    for sinais, labels in treino_loader:

        sinais = sinais.to(device)
        labels = labels.to(device)

        # Zera os gradientes
        otimizador.zero_grad()

        # Forward
        saidas = modelo(sinais)

        # Calcula o erro
        loss = criterio(saidas, labels)

        # Backpropagation
        loss.backward()

        # Atualiza os pesos
        otimizador.step()

        # Acumula o loss
        loss_treino += loss.item()

        # Classe prevista
        _, previsoes = torch.max(saidas, 1)

        total_treino += labels.size(0)

        acertos_treino += (
            previsoes == labels
        ).sum().item()


    # Loss médio do treino
    loss_treino = loss_treino / len(treino_loader)

    # Acurácia do treino
    acc_treino = acertos_treino / total_treino


    # --------------------------------------------------------
    # VALIDAÇÃO
    # --------------------------------------------------------

    modelo.eval()

    loss_validacao = 0
    acertos_validacao = 0
    total_validacao = 0

    # Na validação não precisamos calcular gradientes
    with torch.no_grad():

        for sinais, labels in validacao_loader:

            sinais = sinais.to(device)
            labels = labels.to(device)

            saidas = modelo(sinais)

            loss = criterio(saidas, labels)

            loss_validacao += loss.item()

            _, previsoes = torch.max(saidas, 1)

            total_validacao += labels.size(0)

            acertos_validacao += (
                previsoes == labels
            ).sum().item()


    loss_validacao = (
        loss_validacao / len(validacao_loader)
    )

    acc_validacao = (
        acertos_validacao / total_validacao
    )


    # --------------------------------------------------------
    # SALVANDO RESULTADOS
    # --------------------------------------------------------

    historico_loss_treino.append(loss_treino)
    historico_loss_validacao.append(loss_validacao)

    historico_acc_treino.append(acc_treino)
    historico_acc_validacao.append(acc_validacao)


    # --------------------------------------------------------
    # RESULTADOS DA ÉPOCA
    # --------------------------------------------------------

    print(
        f"Época {epoca+1}/{NUM_EPOCHS} | "
        f"Loss Treino: {loss_treino:.4f} | "
        f"Acc Treino: {acc_treino:.4f} | "
        f"Loss Val: {loss_validacao:.4f} | "
        f"Acc Val: {acc_validacao:.4f}"
    )

    # ========================================================
    # EARLY STOPPING
    # ========================================================

    # Verifica se o modelo melhorou na validação
    if loss_validacao < melhor_val_loss:

        melhor_val_loss = loss_validacao

        melhor_epoca = epoca + 1

        # Guarda os pesos do melhor modelo
        melhor_modelo = copy.deepcopy(modelo.state_dict())

        # Reinicia o contador
        epocas_sem_melhora = 0

    else:

        # Não houve melhora
        epocas_sem_melhora += 1

        print(
            f"Sem melhora: "
            f"{epocas_sem_melhora}/{PATIENCE}"
        )


    # Verifica se atingiu a paciência
    if epocas_sem_melhora >= PATIENCE:

        print("\nEARLY STOPPING ATIVADO!")

        print(f"Treinamento interrompido na época {epoca+1}")

        break


# ============================================================
# RECUPERANDO O MELHOR MODELO
# ============================================================

if melhor_modelo is not None:
    modelo.load_state_dict(melhor_modelo)

print("\nTreinamento concluído!")

print(f"Total de épocas executadas: {len(historico_loss_treino)}")

print(f"Melhor época: {melhor_epoca}")

print(f"Melhor Loss de validação: {melhor_val_loss:.4f}")

In [ ]:
# ============================================================
# 9. GRÁFICO DO LOSS
# ============================================================

epocas = range(1, len(historico_loss_treino) + 1)

plt.figure(figsize=(8, 5))

plt.plot(
    epocas,
    historico_loss_treino,
    label="Treino"
)

plt.plot(
    epocas,
    historico_loss_validacao,
    label="Validação"
)

plt.axvline(
    x=melhor_epoca,
    color="green",
    linestyle="--",
    label=f"Melhor época ({melhor_epoca})"
)

plt.xlabel("Época")
plt.ylabel("Loss")
plt.title("Loss - Treino x Validação")

plt.legend()
plt.grid()

plt.savefig(PASTA_RESULTADOS / f"{PREFIXO}_loss.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ============================================================
# 10. GRÁFICO DA ACURÁCIA
# ============================================================

plt.figure(figsize=(8, 5))

plt.plot(
    epocas,
    historico_acc_treino,
    label="Treino"
)

plt.plot(
    epocas,
    historico_acc_validacao,
    label="Validação"
)

plt.xlabel("Época")
plt.ylabel("Acurácia")
plt.title("Acurácia - Treino x Validação")

plt.legend()
plt.grid()

plt.savefig(PASTA_RESULTADOS / f"{PREFIXO}_acuracia.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ============================================================
# 11. MATRIZ DE CONFUSÃO E MÉTRICAS (validação e teste)
# ============================================================
# Matriz em porcentagem por linha: cada linha é a classe real e soma
# 100%, então a diagonal é o recall (acerto) de cada classe. Entre
# parênteses, a quantidade de exames.
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

tempo_treino = time.time() - inicio_treino
print(f"Tempo de treinamento: {tempo_treino/60:.1f} min")


def prever(loader):
    modelo.eval()
    reais, previstos = [], []
    with torch.no_grad():
        for sinais, labels in loader:
            saidas = modelo(sinais.to(device))
            _, previsoes = torch.max(saidas, 1)
            reais.extend(labels.numpy())
            previstos.extend(previsoes.cpu().numpy())
    return reais, previstos


def desenhar_matriz(matriz, nome):
    porcentagem = matriz / np.maximum(matriz.sum(axis=1, keepdims=True), 1) * 100
    fig, eixo = plt.subplots(figsize=(7, 6))
    imagem = eixo.imshow(porcentagem, cmap="Blues", vmin=0, vmax=100)
    fig.colorbar(imagem, ax=eixo, label="% da classe real")
    eixo.set_xticks(range(len(CLASSES)), CLASSES)
    eixo.set_yticks(range(len(CLASSES)), CLASSES)
    for i in range(len(CLASSES)):
        for j in range(len(CLASSES)):
            cor = "white" if porcentagem[i, j] > 50 else "black"
            eixo.text(j, i, f"{porcentagem[i, j]:.1f}%\n({matriz[i, j]})",
                      ha="center", va="center", color=cor, fontsize=11)
    eixo.set_xlabel("Classe prevista")
    eixo.set_ylabel("Classe real")
    eixo.set_title(f"Matriz de Confusão - {nome}\n{TAREFA} - {ARQUITETURA} - {AUGMENTATION}")
    fig.tight_layout()
    fig.savefig(PASTA_RESULTADOS / f"{PREFIXO}_matriz_{nome.lower()}.png", dpi=150, bbox_inches="tight")
    plt.show()


metricas = {}
for nome, loader in [("Validacao", validacao_loader), ("Teste", teste_loader)]:
    reais, previstos = prever(loader)
    matriz = confusion_matrix(reais, previstos, labels=list(range(len(CLASSES))))
    desenhar_matriz(matriz, nome)
    print(nome)
    print(classification_report(reais, previstos, labels=list(range(len(CLASSES))),
                                target_names=CLASSES, digits=4, zero_division=0))
    metricas[nome.lower()] = {
        "acuracia": accuracy_score(reais, previstos),
        "f1_macro": f1_score(reais, previstos, average="macro"),
        "f1_ponderado": f1_score(reais, previstos, average="weighted"),
        "por_classe": classification_report(reais, previstos, labels=list(range(len(CLASSES))),
                                            target_names=CLASSES, output_dict=True,
                                            zero_division=0),
        "matriz_confusao": matriz.tolist(),
        "classes": CLASSES,
    }

In [ ]:
# ============================================================
# 12. SALVANDO OS RESULTADOS (mesmo formato dos treinos em imagem)
# ============================================================

idx = melhor_epoca - 1
resultado = {
    "dataset": NOME_DATASET,
    "tarefa": TAREFA,
    "classes_finas": TAREFAS[TAREFA],
    "arquitetura": ARQUITETURA,
    "augmentation": AUGMENTATION,
    "contagem_classes": contagem_classes,
    "melhor_epoca": melhor_epoca,
    "epocas_executadas": len(historico_loss_treino),
    "train_acc": historico_acc_treino[idx],
    "train_loss": historico_loss_treino[idx],
    "val_acc": historico_acc_validacao[idx],
    "val_loss": historico_loss_validacao[idx],
    "tempo_min": round(tempo_treino / 60, 2),
    "validacao": metricas["validacao"],
    "teste": metricas["teste"],
    "historico": {
        "loss_treino": historico_loss_treino,
        "loss_validacao": historico_loss_validacao,
        "acc_treino": historico_acc_treino,
        "acc_validacao": historico_acc_validacao,
    },
}

# Salva no Google Drive (o disco do Colab é apagado ao fim da sessão)
with open(PASTA_RESULTADOS / f"{PREFIXO}.json", "w") as f:
    json.dump(resultado, f, indent=2)

# Pesos do melhor modelo (a melhor época já foi carregada no fim do treino)
torch.save(modelo.state_dict(), PASTA_RESULTADOS / f"{PREFIXO}.pt")

print("Arquivos salvos em", PASTA_RESULTADOS)
for arquivo in sorted(PASTA_RESULTADOS.glob(f"{PREFIXO}*")):
    print("  ", arquivo.name)
print(f"Teste: acurácia {metricas['teste']['acuracia']:.4f} | F1 macro {metricas['teste']['f1_macro']:.4f}")

print({k: v for k, v in resultado.items() if k not in ("historico", "contagem_classes", "validacao", "teste")})